# 02 — Zero-shot and few-shot baselines

**Purpose:** Evaluate the unchanged Qwen model in Urdu and English; few-shot demonstrations come ONLY from a training split.

**Platform:** Kaggle or Colab (Python + GPU for model stages). Never use held-out test data for model selection.

**Start here:** The first code cell clones the public GitHub repository automatically when needed. Enable Internet in Kaggle/Colab.

In [ ]:
# 0. CLONE + ENVIRONMENT SETUP — RUN FIRST (Kaggle/Colab).
# Internet must be ON. The FIRST run may install libraries and STOP for a kernel restart.
# After restarting the kernel, rerun this cell; it will skip installation if versions match.
from pathlib import Path
import os, sys, subprocess, importlib.metadata as metadata

REPO_URL = "https://github.com/muhammad-omer-dev/nlp_research_proj.git"
PROJECT_FOLDER = "Urdu_Qwen_Research_Notebooks"
PULL_LATEST = False  # Set True after pushing changes to GitHub; requires a clean Git checkout.

if Path('/kaggle/working').is_dir():
    WORKSPACE = Path('/kaggle/working')
elif Path('/content').is_dir():
    WORKSPACE = Path('/content')
else:
    WORKSPACE = Path.cwd()
REPO_DIR = WORKSPACE / 'nlp_research_proj'
ROOT = REPO_DIR / PROJECT_FOLDER

if (REPO_DIR / '.git').is_dir():
    if PULL_LATEST:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    if REPO_DIR.exists():
        raise RuntimeError(f'{REPO_DIR} exists but is not a Git clone. Rename/remove it after backing up outputs.')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
if not (ROOT / 'requirements.txt').is_file() or not (ROOT / 'project_utils.py').is_file():
    raise FileNotFoundError(f'Expected complete project at {ROOT}. Check the GitHub folder layout.')

# This environment was selected for a stable Transformers 4 training API.
if not ((3, 10) <= sys.version_info[:2] <= (3, 12)):
    raise RuntimeError(f'Python {sys.version.split()[0]} not in tested support range 3.10–3.12.')
REQUIRED = {
    'transformers': '4.51.3',
    'peft': '0.15.2',
    'accelerate': '1.6.0',
    'datasets': '3.5.0',
    'tokenizers': '0.21.1',
    'huggingface-hub': '0.30.2',
    'safetensors': '0.5.3',
    'scikit-learn': '1.6.1',
    'pandas': '2.2.3',
    'numpy': '1.26.4',
    'matplotlib': '3.10.1',
    'psutil': '7.0.0',
    'ipywidgets': '8.1.6',
    'nbformat': '5.10.4',
    'pytest': '8.3.5',
}

def installed_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None

mismatch = {name: (installed_version(name), version)
            for name, version in REQUIRED.items()
            if installed_version(name) != version}
torchao = installed_version('torchao')
if mismatch or torchao is not None:
    print('Installing the project-locked environment; CUDA-enabled PyTorch is left untouched.')
    if mismatch:
        print('Packages to align:', mismatch)
    if torchao is not None:
        print('Removing optional incompatible torchao:', torchao)
        subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT / 'requirements.txt'),
                    '--upgrade-strategy', 'only-if-needed'], check=True)
    raise RuntimeError('Installation finished. RESTART THE KAGGLE/COLAB KERNEL, then rerun this FIRST cell. '
                       'Do not train before restarting.')

# No package installation needed: import the shared research code only now.
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from project_utils import *
folders()

# Catch future API drift before downloading a 0.5B model.
import inspect
from transformers import TrainingArguments, Trainer
sig = inspect.signature(TrainingArguments).parameters
needed = ['output_dir', 'group_by_length', 'warmup_ratio', 'eval_strategy']
missing = [name for name in needed if name not in sig]
if missing:
    raise RuntimeError(f'Transformers does not match our 4.51.3 training API. Missing: {missing}')
api = check_training_api()
print('Ready. Project:', ROOT)
print('Transformers:', api['transformers'], '| PEFT:', api['peft'])
print('Trainer token argument:', api['trainer_tokenizer_key'])
print('No installation needed; proceed to the next cell.')


Run `01_preprocess_data` first. Zero-shot means zero labeled examples in the prompt. Few-shot uses selected **training** examples as demonstrations; these are not gradient updates. Keep greedy decoding and the same exact allowed labels throughout all experiments.

In [ ]:
from collections import Counter
BASE_MODEL = cfg()['model_id']
EVAL_SPLIT = 'validation'  # use test only after deciding hyperparameters
LANG = 'ur'
N_FEWSHOT_PER_CLASS = 1
MAX_ROWS = None  # use 20 for a smoke run, None for complete split
labels = task_labels('sib200')
rows = load_task('sib200', LANG, EVAL_SPLIT)
if MAX_ROWS is not None: rows = rows[:MAX_ROWS]
print('Rows:',len(rows),'Labels:',labels)
model, tokenizer = load_model(BASE_MODEL)
print('Model loaded.')

In [ ]:
demonstrations = examples_for('sib200', N_FEWSHOT_PER_CLASS, 42, lang=LANG)
assert all(r['split']=='train' for r in demonstrations)
print('Demonstrations from training only:',Counter(r['label'] for r in demonstrations))
for condition, demos in [('zero_shot',None),('few_shot',demonstrations)]:
    output = infer_rows(model, tokenizer, rows, labels, LANG, cfg()['max_new_tokens'], demos)
    path = ROOT/'outputs/predictions'/f'sib200_{condition}_{LANG}_{EVAL_SPLIT}.csv'
    meta = dict(task='sib200',lang=LANG,split=EVAL_SPLIT,run_id=condition,model_id=BASE_MODEL,
                dataset_sha256=dataset_sha('sib200',LANG,EVAL_SPLIT),
                subset=(MAX_ROWS is not None),n_fewshot_per_class=N_FEWSHOT_PER_CLASS if demos else 0)
    save_prediction_run(path,output,meta)
    print(condition, len(output), 'saved to',path, 'valid',sum(x['valid'] for x in output))
    print('First five:',[(r['reference'],r['prediction'],r['raw_prediction']) for r in output[:5]])

When done, release GPU memory before training. **Do not use validation performance as published final test performance.**

In [ ]:
del model
import torch
if torch.cuda.is_available(): torch.cuda.empty_cache()